## Variáveis de Ambiente

- Este notebook foi feito para ser importado pelos jobs do pipeline MUNIC
- O objetivo é definir as variáveis comuns aos notebooks do pipeline, facilitando a gestão delas e centralizando a definição em um único lugar, evitando erros de digitação ao longo do pipeline

In [0]:
catalog_name = 'munic_catalog'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

# Ledger do pipeline: uma linha por etapa × ano_edicao com o contrato de execução
# (docs/arquitetura/pipeline_manifest.md). As funções que leem/gravam estão no env_functions.
manifest_tbl = f'{catalog_name}.{gold_schema}.pipeline_manifest'

Atualmente usa-se o munic_file_paths somente na ingest_to_landing, nos demais usa-se o referece_year como base para selecionar a tabela a ser processada no pipeline 

In [0]:
landing_folder_path = '/Volumes/munic_catalog/landing/raw_files'
bronze_folder_path = '/Volumes/munic_catalog/bronze/csv_files'

# Parametrização para múltiplos anos ------------------------------------------------
# Edição corrente do pipeline. Ordem de precedência:
#   1. Parâmetro da tarefa no job (Jobs entrega parâmetros via widgets — o valor
#      passado pelo job sobrescreve o padrão; rodando interativo, troca-se no topo);
#   2. VALORES_PADRAO abaixo, quando o dbutils não existe (Jupyter local/testes).
# Mesmo padrão adotado no bronze_quality_profiling (widgets + fallback NameError).
# NOTA: o widget só aparece no topo DEPOIS que esta célula roda NESTE notebook
# (abrir o notebook não executa nada); quando este notebook é chamado via %run,
# o widget nasce no notebook CHAMADOR, não aqui.
VALORES_PADRAO = {"reference_year": "2024"}

try:
    dbutils.widgets.text("reference_year", VALORES_PADRAO["reference_year"],
                         "Edição MUNIC a processar (ex.: 2024)")
    reference_year = int(dbutils.widgets.get("reference_year"))
except NameError:
    reference_year = int(VALORES_PADRAO["reference_year"])

# URL dos arquivos de origem. E define quais anos devem ser puxados  
munic_source_url_map = {
    2019: "https://ftp.ibge.gov.br/Perfil_Municipios/2019/Base_de_Dados/Base_MUNIC_2019_20210817.xlsx",
    2024: "https://ftp.ibge.gov.br/Perfil_Municipios/2024/Base_de_Dados/Base_MUNIC_2024_20251107.xlsx"
    # Adicionar outros anos conforme a necessidade
}

# Fail-fast: a edição escolhida precisa estar registrada no mapa de URLs acima
if reference_year not in munic_source_url_map:
    raise ValueError(
        f"Edição {reference_year} não registrada em munic_source_url_map "
        f"(disponíveis: {sorted(munic_source_url_map)}). Adicione a URL da edição "
        "no mapa (1 linha) ou corrija o parâmetro reference_year."
    )

# Caminho do arquivo de cada edição na landing (derivado da URL, sem duplicação manual)
munic_file_paths = {
    year: f"{landing_folder_path}/{url.split('/')[-1]}"
    for year, url in munic_source_url_map.items()
}

# Variáveis da edição corrente (reference_year), consumidas por landing_to_bronze e bronze_to_silver.
# Este bloco só escolhe QUAL arquivo processar. A detecção de abas/colunas (célula abaixo)
# não lê nada daqui, ela só recebe os nomes do XLSX da edição selecionada na váriavle reference_year.
munic_source_url = munic_source_url_map[reference_year]
munic_file_name  = munic_source_url.split("/")[-1]
munic_file       = f"{landing_folder_path}/{munic_file_name}"


In [0]:
# --- Overrides por edição (escape opcional) -----------------------------------
# A LÓGICA de detecção (detect_munic_sheets / detect_identity_renames) está no
# env_functions; aqui ficam apenas os overrides por edição, chaveados por reference_year.
# No dia a dia ficam VAZIOS — preencha apenas a edição problemática.
# São consultados pelo landing_to_bronze antes de chamar a detecção automática.
munic_sheet_overrides = {}            # ex.: {2019: {"dict": "Dicionário", "info": "Comunicção e informática", "gov": "Governança"}}
munic_identity_rename_overrides = {}  # ex.: {2019: {"codmun": "cod_munic", "nome_munic": "desc_mun", "classe_pop": "faixa_populacao"}}